# Pump Health Monitoring - Exploratory Data Analysis
This notebook explores the pump sensor data and performs initial analysis.

In [ ]:
import sys
sys.path.append('../src')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from data_ingestion import PumpDataIngestion
from feature_engineering import PumpFeatureEngineering

# Set style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')
%matplotlib inline

## 1. Load Data

In [ ]:
# Initialize data ingestion
ingestion = PumpDataIngestion(data_path='../data/raw')

# Load sensor data
df = ingestion.load_sensor_data('pump_sensor_data.csv')

print(f"Data shape: {df.shape}")
print(f"\nColumns: {df.columns.tolist()}")
df.head()

## 2. Data Overview

In [ ]:
# Basic statistics
df.describe()

In [ ]:
# Check for missing values
print("Missing values:")
print(df.isnull().sum())

In [ ]:
# Data types
print("Data types:")
print(df.dtypes)

## 3. Sensor Data Visualization

In [ ]:
# Plot sensor trends
fig, axes = plt.subplots(3, 2, figsize=(15, 12))
fig.suptitle('Sensor Readings Over Time', fontsize=16)

sensors = ['vibration', 'temperature', 'pressure', 'flow_rate', 'current', 'rpm']

for idx, sensor in enumerate(sensors):
    ax = axes[idx // 2, idx % 2]
    ax.plot(df['timestamp'], df[sensor], linewidth=0.5)
    ax.set_title(sensor.replace('_', ' ').title())
    ax.set_xlabel('Time')
    ax.set_ylabel('Value')
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 4. Distribution Analysis

In [ ]:
# Distribution plots
fig, axes = plt.subplots(3, 2, figsize=(15, 12))
fig.suptitle('Sensor Data Distributions', fontsize=16)

for idx, sensor in enumerate(sensors):
    ax = axes[idx // 2, idx % 2]
    ax.hist(df[sensor], bins=50, edgecolor='black', alpha=0.7)
    ax.set_title(sensor.replace('_', ' ').title())
    ax.set_xlabel('Value')
    ax.set_ylabel('Frequency')
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 5. Correlation Analysis

In [ ]:
# Correlation matrix
corr_matrix = df[sensors + ['rul']].corr()

plt.figure(figsize=(10, 8))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', center=0,
            square=True, linewidths=1, cbar_kws={"shrink": 0.8})
plt.title('Sensor Correlation Matrix', fontsize=14)
plt.tight_layout()
plt.show()

## 6. Health Status Analysis

In [ ]:
# Health status distribution
health_counts = df['health_status'].value_counts()

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Pie chart
axes[0].pie(health_counts, labels=health_counts.index, autopct='%1.1f%%', startangle=90)
axes[0].set_title('Health Status Distribution')

# Bar chart
health_counts.plot(kind='bar', ax=axes[1], color=['green', 'orange', 'red'])
axes[1].set_title('Health Status Count')
axes[1].set_xlabel('Status')
axes[1].set_ylabel('Count')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 7. RUL Analysis

In [ ]:
# RUL over time
fig, axes = plt.subplots(2, 1, figsize=(15, 10))

# RUL trend
axes[0].plot(df['timestamp'], df['rul'], linewidth=1)
axes[0].set_title('Remaining Useful Life (RUL) Over Time')
axes[0].set_xlabel('Time')
axes[0].set_ylabel('RUL (days)')
axes[0].grid(True, alpha=0.3)
axes[0].axhline(y=30, color='r', linestyle='--', label='Critical Threshold')
axes[0].axhline(y=90, color='orange', linestyle='--', label='Warning Threshold')
axes[0].legend()

# RUL distribution
axes[1].hist(df['rul'], bins=50, edgecolor='black', alpha=0.7)
axes[1].set_title('RUL Distribution')
axes[1].set_xlabel('RUL (days)')
axes[1].set_ylabel('Frequency')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 8. Sensor Relationships

In [ ]:
# Scatter plots for key relationships
fig, axes = plt.subplots(2, 2, figsize=(15, 12))

# Vibration vs Temperature
axes[0, 0].scatter(df['vibration'], df['temperature'], alpha=0.3, s=1)
axes[0, 0].set_xlabel('Vibration')
axes[0, 0].set_ylabel('Temperature')
axes[0, 0].set_title('Vibration vs Temperature')
axes[0, 0].grid(True, alpha=0.3)

# Pressure vs Flow Rate
axes[0, 1].scatter(df['pressure'], df['flow_rate'], alpha=0.3, s=1)
axes[0, 1].set_xlabel('Pressure')
axes[0, 1].set_ylabel('Flow Rate')
axes[0, 1].set_title('Pressure vs Flow Rate')
axes[0, 1].grid(True, alpha=0.3)

# Current vs RPM
axes[1, 0].scatter(df['current'], df['rpm'], alpha=0.3, s=1)
axes[1, 0].set_xlabel('Current')
axes[1, 0].set_ylabel('RPM')
axes[1, 0].set_title('Current vs RPM')
axes[1, 0].grid(True, alpha=0.3)

# Vibration vs RUL
axes[1, 1].scatter(df['vibration'], df['rul'], alpha=0.3, s=1)
axes[1, 1].set_xlabel('Vibration')
axes[1, 1].set_ylabel('RUL (days)')
axes[1, 1].set_title('Vibration vs RUL')
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 9. Summary Statistics by Health Status

In [ ]:
# Group by health status
grouped = df.groupby('health_status')[sensors].mean()
print("Mean sensor values by health status:")
print(grouped)

# Visualize
grouped.T.plot(kind='bar', figsize=(12, 6))
plt.title('Mean Sensor Values by Health Status')
plt.xlabel('Sensor')
plt.ylabel('Mean Value')
plt.xticks(rotation=45)
plt.legend(title='Health Status')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 10. Conclusions

Key findings from the exploratory analysis:
1. Sensor data shows expected patterns for pump degradation over time
2. Vibration and temperature show positive correlation with degradation
3. Pressure anomalies are visible in the data
4. RUL decreases linearly with operating time
5. Clear distinction between health status categories

Next steps:
- Feature engineering to create more predictive features
- Model training for RUL prediction
- Deploy dashboard for real-time monitoring